### Pipeline ML afin de vérifier les résultats du papier : Cascading Machine Learning to Attack Bitcoin Anonymity

#### Importation des modules nécessaires

In [17]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

# Importation des fonctions depuis utils.py
from utils import evaluate_classifier, prepare_and_init_model, run_cascade_layer

print("Modules chargés avec succès.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Modules chargés avec succès.


#### Chargement des données  
En l'absence des données réelles pour l'instant, nous générons des données synthétiques (Mock Data) pour valider le pipeline ML.  
Première cellule : Mock data relativement complexe pour pouvoir observer une différence entre la baseline et le final. Sur les petites données, les scores étaient identiques et laissaient penser à une erreur dans la logique du code.   
Deuxième cellule : Mock data initiales pour tester le bon fonctionnement du pipeline. On peut décommenter les lignes concernant motif2.

In [18]:
# --- GÉNÉRATION DE DONNÉES SEMI-RÉALISTES (LOGIQUE "PIÈGE") ---

# Configuration
n_entities = 1000
classes = ['Exchange', 'Mixer', 'Gambling', 'Marketplace']
np.random.seed(42) 

# 1. GÉNÉRATION DES ENTITÉS (Le "Flou" pour la Baseline)
# Stratégie : On rend le 'total_balance' trompeur. 
# Normalement un Exchange est riche et un Mixer est moyen.
# Ici, on va mélanger les distributions pour que la Baseline hésite.

data_ent = []
for i in range(n_entities):
    label = np.random.choice(classes)
    
    # Logique de solde ambiguë
    if label == 'Exchange':
        # Majorité riches, mais certains pauvres (piège)
        balance = np.random.uniform(1000, 10000) if np.random.rand() > 0.2 else np.random.uniform(50, 200)
        n_tx = np.random.randint(1000, 5000)
    elif label == 'Mixer':
        # Beaucoup ressemblent à du Gambling (solde moyen)
        balance = np.random.uniform(100, 1000)
        n_tx = np.random.randint(500, 2000)
    elif label == 'Gambling':
        balance = np.random.uniform(50, 500)
        n_tx = np.random.randint(100, 1000)
    else: # Marketplace
        balance = np.random.uniform(10, 300)
        n_tx = np.random.randint(10, 200)
        
    data_ent.append([i, f"Ent_{i}", balance, np.random.randint(5, 50), n_tx, label])

df_entity = pd.DataFrame(data_ent, columns=['entity_id', 'entity_name', 'total_balance', 'n_addresses', 'n_transactions', 'label'])
ignore_cols_entity = ['entity_id', 'entity_name']

print("Répartition des classes :")
print(df_entity['label'].value_counts())


# 2. GÉNÉRATION DES ADRESSES (La "Vérité" pour la Cascade)
# Ici, on met des patterns très clairs. Si le modèle regarde l'adresse, il doit deviner la classe à 100%.

data_add = []
for _, row in df_entity.iterrows():
    # Nombre d'adresses par entité (variable)
    n_addr = np.random.randint(5, 25) 
    
    for _ in range(n_addr):
        # CARACTÉRISTIQUES DISCRIMINANTES (La signature comportementale)
        
        # SIBLINGS (Frères) : Les Mixers créent des clusters énormes
        if row['label'] == 'Mixer':
            n_siblings = np.random.randint(20, 50) # Signature forte
            is_unique = 1 # Ne réutilise jamais
        elif row['label'] == 'Exchange':
            n_siblings = np.random.randint(0, 5)   # Peu de clustering direct
            is_unique = 0 # Réutilise souvent les adresses de dépôt
        else:
            n_siblings = np.random.randint(0, 3)
            is_unique = np.random.randint(0, 2)
            
        # BALANCE ADRESSE : Les Gambling sites ont des montants fixes/ronds (simulé ici par std faible)
        if row['label'] == 'Gambling':
            addr_bal = np.random.normal(50, 5) 
        else:
            addr_bal = np.random.exponential(50)

        data_add.append([
            f"A_{np.random.randint(1e9)}", 
            row['entity_id'], 
            addr_bal, 
            np.random.rand()*100, 
            is_unique, 
            n_siblings,
            row['label'] # Nécessaire pour le split stratifié
        ])

df_address = pd.DataFrame(data_add, columns=['address_id', 'entity_id', 'addr_recv_amount', 'addr_balance', 'is_unique', 'n_siblings', 'label'])
ignore_cols_add = ['entity_id', 'address_id', 'label']


# 3. GÉNÉRATION DES MOTIFS (Bonus de précision)
# On simule un "Motif 1" qui détecte les boucles (typique des Mixers et Marketplaces)

data_m1 = []
for _, row in df_entity.iterrows():
    # Signature Motif : Force du cycle
    if row['label'] in ['Mixer', 'Marketplace']:
        cycle_strength = np.random.uniform(0.7, 1.0) # Fort
    else:
        cycle_strength = np.random.uniform(0.0, 0.4) # Faible
        
    data_m1.append([f"M1_{row['entity_id']}", row['entity_id'], cycle_strength, row['label']])

df_motif1 = pd.DataFrame(data_m1, columns=['motif_id', 'entity_id', 'cycle_strength', 'label'])
ignore_cols_m1 = ['entity_id', 'motif_id', 'label']

# Pas de Motif 2 pour l'instant pour alléger
df_motif2 = pd.DataFrame(columns=['entity_id', 'label']) # Vide mais existant pour ne pas casser le code

print(f"\nDonnées générées : {len(df_entity)} Entités, {len(df_address)} Adresses.")
print("Scénario : Les 'statistiques globales' sont floues, mais les 'comportements locaux' (adresses) sont clairs.")

Répartition des classes :
label
Exchange       265
Gambling       251
Mixer          249
Marketplace    235
Name: count, dtype: int64

Données générées : 1000 Entités, 14451 Adresses.
Scénario : Les 'statistiques globales' sont floues, mais les 'comportements locaux' (adresses) sont clairs.


#### Mock Data originelles

In [16]:
# Création du df_entity (inventé car pas de données pour l'instant)
data_entity = {
    'entity_id': [101, 102, 103, 104, 105],
    'entity_name': ['Kraken', 'BitMixer', 'SatoshiDice', 'Binance', 'SilkRoad'],
    'total_balance': [5000.5, 120.0, 300.5, 8000.0, 50.0],
    'n_addresses': [100, 20, 500, 150, 10],             
    'n_transactions': [5000, 400, 12000, 7000, 50],      
    'label': ['Exchange', 'Mixer', 'Gambling', 'Exchange', 'Marketplace']
}
df_entity = pd.DataFrame(data_entity)

# Stockages des variables inutiles de df_entity (et sa variante df_final) pour l'entraînement des modèles
ignore_cols_entity = ['entity_id', 'entity_name']

# Création du df_address (idem)
data_address = {
    'address_id': [
        'A1', 'A2', 'A3', 'A4',       # Kraken (Exchange)
        'B1', 'B2', 'B3',             # BitMixer (Mixer)
        'C1', 'C2', 'C3',             # SatoshiDice (Gambling) 
        'D1', 'D2', 'D3', 'D4',       # Binance (Exchange)
        'E1', 'E2'                    # SilkRoad (Marketplace)
    ],
    'addr_recv_amount': np.random.rand(16) * 1000, 
    'addr_balance': np.random.rand(16) * 100,
    'is_unique': [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0],
    'n_siblings': np.random.randint(0, 20, 16),
    'entity_id': [
        101, 101, 101, 101,
        102, 102, 102,
        103, 103, 103,
        104, 104, 104, 104,
        105, 105
    ]
}
df_address = pd.DataFrame(data_address)

# On attribue à chaque adresse le label de son entité
df_address = df_address.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')
# On stocke les variables inutiles de df_address pour l'entraînement des modèles
ignore_cols_add = ['entity_id', 'address_id']

# Création du df_motif1 (idem)
data_motif1 = {
    'motif_id': [f'M1_{i}' for i in range(15)],
    'entity_id': [101, 101, 101, 102, 102, 103, 103, 103, 104, 104, 104, 104, 105, 105, 105],
    'motif_weight': np.random.rand(15),       
    'motif_degree': np.random.randint(1, 5, 15)
}
df_motif1 = pd.DataFrame(data_motif1)

# Attribution à chaque 1_motif du label de son entité
df_motif1 = df_motif1.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')
# Stockages des variables inutiles de df_motif1 pour l'entraînement des modèles
ignore_cols_m1 = ['entity_id', 'motif_id']

# Création du df_motif2 (idem)
data_motif2 = {
    'motif_id': [f'M2_{i}' for i in range(12)], 
    'entity_id': [101, 101, 102, 102, 103, 103, 103, 104, 104, 105, 105, 101],
    'closeness_centrality': np.random.rand(12), 
    'pattern_size': np.random.randint(3, 10, 12) 
}
df_motif2 = pd.DataFrame(data_motif2)
# Attribution à chaque 2_motif du label de son entité
df_motif2 = df_motif2.merge(df_entity[['entity_id', 'label']], on='entity_id', how='left')
# Stockages des variables inutiles de df_motif2 pour l'entraînement des modèles
ignore_cols_m2 = ['entity_id', 'motif_id']

print("Mock data chargées : Entité, Address, Motif1, Motif2.")

Mock data chargées : Entité, Address, Motif1, Motif2.


#### Entraînement du classifieur de référence (Baseline) C_entity​ utilisant uniquement les attributs globaux des entités, sans l'enrichissement par cascade.  


In [20]:
# Entraînement d'un classifier sur le df_entity brut pour servir de comparaison

MODEL_TYPE = "GradientBoosting"
# N_SPLITS = 5 dans le papier
N_SPLITS = 2

# Initialisation du Classifier (paramètres issus du papier)
C_entity, X_ent, y_ent = prepare_and_init_model(df_entity, ignore_cols_entity, model_type=MODEL_TYPE)

# Evaluation du classifier C_entity par Stratified K-Fold Cross Validation
res_baseline = evaluate_classifier(C_entity, X_ent, y_ent, n_splits=N_SPLITS)
baseline_acc = res_baseline["acc_mean"]
baseline_acc_std = res_baseline["acc_std"]
baseline_mcc = res_baseline["mcc_mean"]
baseline_mcc_std = res_baseline["mcc_std"]


Préparation du modèle : GradientBoosting
Features sélectionnées : ['total_balance', 'n_addresses', 'n_transactions']
Accuracy moyenne : 0.90 (+/- 0.02)
MCC moyenne : 0.90 (+/- 0.02)


#### Entraînement des classifiers C_address, C_motif1 et C_motif2, prédiction grâce à ces derniers des classes des adresses, 1_motifs et 2_motifs séléctionnés pour la génération des nouvelles features, création des nouvelles features pour enrichir df_entity.

In [21]:
MODEL_TYPE = 'RandomForest'
TEST_SIZE = 0.3

new_features_add = run_cascade_layer(df_address, 'add', ignore_cols_add, model_type=MODEL_TYPE, test_size=TEST_SIZE)
new_features_motif1 = run_cascade_layer(df_motif1, 'mot1', ignore_cols_m1, model_type=MODEL_TYPE, test_size=TEST_SIZE)
#new_features_motif2 = run_cascade_layer(df_motif2, 'mot2', ignore_cols_m2, model_type=MODEL_TYPE, test_size=TEST_SIZE)

Cascade Layer : 'add'
Split : 10115 train (Set A) / 4336 génération des features (Set B)
Préparation du modèle : RandomForest
Features sélectionnées : ['addr_recv_amount', 'addr_balance', 'is_unique', 'n_siblings']
Accuracy intermédiaire sur Set B : 80.77%
Cascade Layer : 'mot1'
Split : 700 train (Set A) / 300 génération des features (Set B)
Préparation du modèle : RandomForest
Features sélectionnées : ['cycle_strength']
Accuracy intermédiaire sur Set B : 56.67%


#### Phase de Cascade : Fusion des features avec df_entity pour entraîner le modèle final C_final​.

In [22]:
df_final = df_entity.copy()

# Enrichissement de df_entity avec les nouvelles features calculées
df_final = df_final.merge(new_features_add, on='entity_id', how='left')
df_final = df_final.merge(new_features_motif1, on='entity_id', how='left')
#df_final = df_final.merge(new_features_motif2, on='entity_id', how='left')

# Si une entité n'avait aucune adresse dans le Set B (improbable mais peut arrivé sur faux jeu de données),
# Remplacement de tous les nan par 0.0 
cascade_cols = [c for c in df_final.columns if ' as ' in c]
df_final[cascade_cols] = df_final[cascade_cols].fillna(0.0)

MODEL_TYPE = 'GradientBoosting'
# N_SPLITS = 5 dans le papier
N_SPLITS = 2
# Initialisation du Classifier (paramètres issus du papier)
C_final, X_final, y_final = prepare_and_init_model(df_final, ignore_cols_entity, model_type=MODEL_TYPE)

# Evaluation du classifier C_final par Stratified K-Fold Cross Validation
res_final = evaluate_classifier(C_final, X_final, y_final, n_splits=N_SPLITS)
final_acc = res_final["acc_mean"]
final_acc_std = res_final["acc_std"]
final_mcc = res_final["mcc_mean"]
final_mcc_std = res_final["mcc_std"]


res_data = {
    'Mean Accuracy': [baseline_acc, final_acc],
    'Std Deviation': [baseline_acc_std, final_mcc_std],
    'MCC' : [baseline_mcc, final_mcc]
}
df_results = pd.DataFrame(res_data, index=['Baseline', 'Final (Cascade)'])

print("\n Comparaison Baseline / Final")
display(df_results)



Préparation du modèle : GradientBoosting
Features sélectionnées : ['total_balance', 'n_addresses', 'n_transactions', 'add as Exchange', 'add as Gambling', 'add as Marketplace', 'add as Mixer', 'mot1 as Exchange', 'mot1 as Gambling', 'mot1 as Marketplace', 'mot1 as Mixer']
Accuracy moyenne : 0.98 (+/- 0.01)
MCC moyenne : 0.98 (+/- 0.01)

 Comparaison Baseline / Final


,Mean Accuracy,Std Deviation,MCC
Baseline,0.897,0.011000,0.863079
Final (Cascade),0.980,0.005408,0.973511
